In [1]:
import pandas as pd

# Ucitavanje podataka
df = pd.read_csv("products.csv")

# Prikaz prvih 5 redova
print("Prvih 5 proizvoda u tabeli:")
display(df.head())

# Osnovne informacije o tabeli
print("\nStruktura podataka:")
df.info()

Prvih 5 proizvoda u tabeli:


,product ID,Product Title,Merchant ID,Category Label,_Product Code,Number_of_Views,Merchant Rating,Listing Date
0,1,apple iphone 8 plus 64gb silver,1,Mobile Phones,QA-2276-XC,860.0,2.5,5/10/2024
1,2,apple iphone 8 plus 64 gb spacegrau,2,Mobile Phones,KA-2501-QO,3772.0,4.8,12/31/2024
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,3,Mobile Phones,FP-8086-IE,3092.0,3.9,11/10/2024
3,4,apple iphone 8 plus 64gb space grey,4,Mobile Phones,YI-0086-US,466.0,3.4,5/2/2022
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,5,Mobile Phones,NZ-3586-WP,4426.0,1.6,4/12/2023



Struktura podataka:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35311 entries, 0 to 35310
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   product ID       35311 non-null  int64  
 1   Product Title    35139 non-null  object 
 2   Merchant ID      35311 non-null  int64  
 3    Category Label  35267 non-null  object 
 4   _Product Code    35216 non-null  object 
 5   Number_of_Views  35297 non-null  float64
 6   Merchant Rating  35141 non-null  float64
 7    Listing Date    35252 non-null  object 
dtypes: float64(2), int64(2), object(4)
memory usage: 2.2+ MB


In [3]:
# 1. Zadrzavamo samo kolone koje su nam vazne za ovaj zadatak
df_model = df[["Product Title", " Category Label"]].copy()

# 2. Brisemo sve redove gde nedostaje naslov ili kategorija
df_model = df_model.dropna(subset=["Product Title", " Category Label"])

# 3. Proveravamo preostali broj podataka nakon ciscenja
print("Broj podataka nakon brisanja praznih vrednosti:")
print(df_model.shape)

# 4. Da vidimo koliko razlicitih kategorija uopste imamo u datasetu
print("\nBroj proizvoda po kategorijama:")
print(df_model[" Category Label"].value_counts())

Broj podataka nakon brisanja praznih vrednosti:
(35096, 2)

Broj proizvoda po kategorijama:
 Category Label
Fridge Freezers     5470
Washing Machines    4015
Mobile Phones       4002
CPUs                3747
TVs                 3541
Fridges             3436
Dishwashers         3405
Digital Cameras     2689
Microwaves          2328
Freezers            2201
fridge               123
CPU                   84
Mobile Phone          55
Name: count, dtype: int64


In [5]:
# 1. Pretvaramo sve naslove proizvoda u mala slova (standardizacija teksta)
df_model["Product Title"] = df_model["Product Title"].str.lower()

# 2. Pretvaramo sve kategorije tako da pocinju velikim slovom
df_model[" Category Label"] = df_model[" Category Label"].str.title()

# 3. Pravimo recnik za spajanje slicnih kategorija
mapiranje_kategorija = {
    "Mobile Phone": "Mobile Phones",
    "CPU": "CPUs",
    "Fridge": "Fridges"
}

# 4. Primenjujemo mapiranje (ako je neka kategorija u recniku, zamenice je, ostale ostaju iste)
df_model[" Category Label"] = df_model[" Category Label"].replace(mapiranje_kategorija)

# 5. Proveravamo ponovo stanje da vidimo da li su se kategorije spojile
print("Sredjene i ciste kategorije za model:")
display(df_model[" Category Label"].value_counts())

Sredjene i ciste kategorije za model:


 Category Label
Fridge Freezers     5470
Mobile Phones       4057
Washing Machines    4015
Cpus                3747
Fridges             3559
Tvs                 3541
Dishwashers         3405
Digital Cameras     2689
Microwaves          2328
Freezers            2201
Cpu                   84
Name: count, dtype: int64

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Podela na ulazne podatke (X - tekst) i ciljanu klasu (y - kategorija)
X = df_model["Product Title"]
y = df_model[" Category Label"]

# Podela na Train i Test set (80% za trening, 20% za test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Vektorizacija - pretvaranje teksta u brojeve (TG-IDF)
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 3. Model 1: Naivni Bajes (Multinomial Naive Bayes)
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_preds)

# 4. Model 2: Logisticka Regresija
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, lr_preds)

# Prikaz rezultata tacnosti
print("--- Rezultati modelovanja ---")
print(f"Tacnost Naivnog Bajsa: {nb_acc * 100:.2f}%")
print(f"Tacnost Logisticke Regresije: {lr_acc * 100:.2f}%")

--- Rezultati modelovanja ---
Tacnost Naivnog Bajsa: 92.98%
Tacnost Logisticke Regresije: 95.20%


In [10]:
# Generisanje detaljnog izvestaja o klasifikaciji za Logisticke regresiju
print("---Detaljan izvestaj o evaluaciji (Logisticka regresija) ---")
print(classification_report(y_test, lr_preds))

---Detaljan izvestaj o evaluaciji (Logisticka regresija) ---
                  precision    recall  f1-score   support

             Cpu       0.00      0.00      0.00        14
            Cpus       0.98      1.00      0.99       726
 Digital Cameras       0.99      0.99      0.99       535
     Dishwashers       0.92      0.95      0.93       684
        Freezers       0.99      0.87      0.92       422
 Fridge Freezers       0.94      0.93      0.93      1087
         Fridges       0.85      0.91      0.88       731
      Microwaves       0.99      0.96      0.97       464
   Mobile Phones       0.97      0.98      0.98       812
             Tvs       0.98      0.98      0.98       724
Washing Machines       0.97      0.95      0.96       821

        accuracy                           0.95      7020
       macro avg       0.87      0.87      0.87      7020
    weighted avg       0.95      0.95      0.95      7020



c:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Zaključak i evaluacija modela

U ovom projektu uspešno smo kreirali model za predikciju kategorija proizvoda na osnovu njihovog naslova.

Tokom rada realizovane su sledeće faze:
1. **Učitavanje i čišćenje podataka**: Uklonjene su nedostajuće vrednosti i standardizovani su nazivi kategorija (npr. spajanje jednine i množine kod telefona i procesora)
2. **Inženjering karakteristika**: Tekstualni podaci su prebačeni u mala slova i transformisani u numeričke vektore pomoću TF-IDF metodologije (ograničeno na top 5000 karakteristika).
3. **Treniranje i poređenje modela**: Testirana su dva algoritma:
    * Naivni Bajes (Tačnost: 92.98%)
    * Logistička regresija (Tačnost: 95.20%)

**Finalni zaključak**: Logistička regresija je izabrana kao najbolji model sa ukupnom tačnošću od **95.20%**. Detaljan izveštaj o klasifikaciji pokazuje visoke vrednosti F1-skora na svim pojedinačnim kategorijama, što potvrđuje stabilnost i visoku pouzdanost modela.